In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.utils import *

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import lightgbm as lgbm

import optuna as op

In [ ]:
# Read data files
x, y, x_test = load_processed_data()

# Test train split from sklearn model selection
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Optuna objective function to find the optimal value for each of the features specified to find the highest auc score
def objecitve(trial):
    params = {

        # Static values and declarations
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": 42,

        # Dynamic values 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),          # Defines that it is a 2 state problem 
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),                     # Size of each tree
        "max_depth": trial.suggest_int("max_depth", 3, 12),                         # Limits tree depth
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),       # Minimum samples that are needed in one of the leaves
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),                    # Row sampling to improve generalization
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),      # Feature Sampling
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),                   # Regularization to penalize complexity
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),                 # Regularization to penalize larger weight values

    }

    # LGBM Model configuration
    OPmodel = lgbm.LGBMClassifier(
        **params,
        n_estimators=5000,                  # Number of trees that can be generated
    )

    # Training model
    OPmodel.fit(
        x_train,                        # Training Features
        y_train,                        # Solutions to those features
        eval_set=[(x_val, y_val)],      # Evaluate performance for validation data
        eval_metric="auc",               # Specify ROC AUC as the performance metric
        callbacks=[
            lgbm.early_stopping(100),
            lgbm.log_evaluation(0)
        ]
    )

    # Generating validation prediction
    # Makes predictions and classifies them as probabilities over flat integers
    # Also uses [:,1] as it returns two probabilitiy rows for no pit vs pit
    val_prediction = OPmodel.predict_proba(x_val)[:,1]       

    # Calculates auc score to evaluate performance of the model and prints
    auc = generate_auc(y_val, val_prediction)

    return auc

# Create and run the optuna study
study = op.create_study(direction="maximize")

# optimize the given features in the objective function and it runs this optimization up to 30 times
study.optimize(
    objecitve,
    n_trials=30
)

# Output confirmation
print("Best AUC Score: ", study.best_value)
print("Optimal Parameters: ")
print(study.best_params)

# Finalize model declaration using new parameters
model = lgbm.LGBMClassifier(
    **study.best_params,
    objective="binary",
    metric = "auc",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=42,
    n_estimators=5000
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgbm.early_stopping(100),
        lgbm.log_evaluation(100)
    ]
)

# Validation Prediction
validation_prediction = generate_probability_predict(model, x_val)

# AUC score test
generate_auc(y_val, validation_prediction)

predictions = generate_probability_predict(model, x_test)

[I 2026-05-17 16:25:51,733] A new study created in memory with name: no-name-2f1ce522-575a-4e24-9af1-de919e22b731


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2123]	valid_0's auc: 0.949431


[I 2026-05-17 16:26:20,254] Trial 0 finished with value: 0.9494311170528003 and parameters: {'learning_rate': 0.03737679909727055, 'num_leaves': 85, 'max_depth': 7, 'min_child_samples': 40, 'subsample': 0.8584638062247886, 'colsample_bytree': 0.6841017908077892, 'reg_alpha': 8.894070882122909, 'reg_lambda': 8.687773575347805}. Best is trial 0 with value: 0.9494311170528003.


AUC: 0.9494
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[5000]	valid_0's auc: 0.947679


[I 2026-05-17 16:27:09,489] Trial 1 finished with value: 0.9476787739453121 and parameters: {'learning_rate': 0.012449137251883097, 'num_leaves': 41, 'max_depth': 5, 'min_child_samples': 72, 'subsample': 0.879657320821815, 'colsample_bytree': 0.7152199695030963, 'reg_alpha': 6.496143518906652, 'reg_lambda': 8.969985988071327}. Best is trial 0 with value: 0.9494311170528003.


...

[I 2026-05-17 16:42:39,941] Trial 28 finished with value: 0.9496671168199586 and parameters: {'learning_rate': 0.019554648770988208, 'num_leaves': 91, 'max_depth': 6, 'min_child_samples': 82, 'subsample': 0.9721420626204798, 'colsample_bytree': 0.6330954294013017, 'reg_alpha': 3.3472114010012044, 'reg_lambda': 2.499707886645555}. Best is trial 23 with value: 0.949945019956021.


AUC: 0.9497
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1993]	valid_0's auc: 0.949768


[I 2026-05-17 16:43:01,826] Trial 29 finished with value: 0.9497676925508888 and parameters: {'learning_rate': 0.031683897185763664, 'num_leaves': 68, 'max_depth': 7, 'min_child_samples': 51, 'subsample': 0.999210814868383, 'colsample_bytree': 0.6789291170081413, 'reg_alpha': 0.6795805610189251, 'reg_lambda': 3.7252051221189966}. Best is trial 23 with value: 0.949945019956021.


AUC: 0.9498
Best AUC Score:  0.949945019956021
Optimal Parameters: 
{'learning_rate': 0.020541079651710786, 'num_leaves': 83, 'max_depth': 8, 'min_child_samples': 89, 'subsample': 0.9997820637792558, 'colsample_bytree': 0.6348560759485127, 'reg_alpha': 2.42890524945383, 'reg_lambda': 3.852578357505773}
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.935634
[200]	valid_0's auc: 0.941097
[300]	valid_0's auc: 0.943684
[400]	valid_0's auc: 0.94533
[500]	valid_0's auc: 0.946366
[600]	valid_0's auc: 0.947107
[700]	valid_0's auc: 0.947658
[800]	valid_0's auc: 0.948058
[900]	valid_0's auc: 0.948369
[1000]	valid_0's auc: 0.948684
[1100]	valid_0's auc: 0.948877
[1200]	valid_0's auc: 0.94908
[1300]	valid_0's auc: 0.949242
[1400]	valid_0's auc: 0.949381
[1500]	valid_0's auc: 0.949476
[1600]	valid_0's auc: 0.949578
[1700]	valid_0's auc: 0.949639
[1800]	valid_0's auc: 0.949691
[1900]	valid_0's auc: 0.949719
[2000]	valid_0's auc: 0.949791
[2100]	valid_0's auc: 0.9

0.949945019956021

In [7]:
predictions = generate_probability_predict(model, x_test)
generate_submission(predictions, "../submissions/lgbm_optuna_sub.csv")

,id,PitNextLap
0,439140,0.004591
1,439141,0.004686
2,439142,0.004380
3,439143,0.180671
4,439144,0.881736
...,...,...
188160,627300,0.015786
188161,627301,0.582041
188162,627302,0.780084
188163,627303,0.827216
